In [1]:
using Gridap        
using GridapGmsh
using Gridap.Geometry
using Gridap.TensorValues
using Plots
using LinearAlgebra
using  Gridap.Fields
using  Gridap.CellData
using  Gridap.ReferenceFEs  
using  Gridap.Fields

In [2]:
model = GmshDiscreteModel("UniaxialComp_RockSquareDomain.msh")
writevtk(model,"UniaxialComp_RockSquareDomain")

Info    : Reading 'UniaxialComp_RockSquareDomain.msh'...
Info    : 17 entities
Info    : 29546 nodes
Info    : 58268 elements
Info    : Done reading 'UniaxialComp_RockSquareDomain.msh'


3-element Vector{Vector{String}}:
 ["UniaxialComp_RockSquareDomain_0.vtu"]
 ["UniaxialComp_RockSquareDomain_1.vtu"]
 ["UniaxialComp_RockSquareDomain_2.vtu"]

In [3]:
const ν = 0.21
const E = 36200
const G = E/(2*(1+ν))

14958.677685950413

In [4]:
const λ = (E*ν)/((1+ν)*(1-2*ν))
const μ = G
const κ = λ + μ

25790.82359646623

In [5]:
D = 2
G0 = 1e8
r = 1.5 
τ = 1e-6 
R = 0.01
fₜ = 10 
delT1 = 1e-4

0.0001

In [6]:
Fric_ang = 34*π/180
cϕ = cos(Fric_ang)
tϕ = tan(Fric_ang)
coh = 15 
k1 = 1.0
k2 = 1.0
k3 = 1.0

1.0

In [7]:
σ(ε)= λ*tr(ε)*one(ε) + 2*μ*ε

σ (generic function with 1 method)

In [8]:
degree = 2
Ω = Triangulation(model)
dΩ = Measure(Ω,degree)

Measure()

In [9]:
labels = get_face_labeling(model)

FaceLabeling:
 0-faces: 29546
 1-faces: 87766
 2-faces: 58220
 tags: 5
 entities: 17

In [10]:
Gr = get_grid(model)
nodes = get_node_coordinates(Gr)
Nₑ, Nₙ = num_cells(model), num_nodes(model)
nodeCoordX, nodeCoordY = [nodes[i][1] for i in 1:Nₙ], [nodes[i][2] for i in 1:Nₙ]
elem = get_cell_node_ids(Gr)

58220-element Gridap.Arrays.Table{Int32, Vector{Int32}, Vector{Int32}}:
 [11354, 11401, 11417]
 [11271, 11272, 11275]
 [1309, 2815, 28253]
 [12871, 12920, 25369]
 [5134, 5171, 27409]
 [2515, 5868, 27211]
 [4821, 5867, 27210]
 [9878, 9880, 10038]
 [9277, 9537, 11848]
 [5257, 25813, 25873]
 [24350, 24380, 26622]
 [11271, 11275, 11276]
 [9191, 23531, 23532]
 ⋮
 [12715, 26980, 27897]
 [12870, 12871, 12920]
 [12938, 15662, 29489]
 [15139, 26685, 27334]
 [15475, 15618, 15626]
 [16665, 16666, 16668]
 [16782, 26884, 28615]
 [21794, 21796, 27052]
 [24217, 27113, 28033]
 [24357, 24380, 24389]
 [24359, 24381, 24397]
 [25473, 25474, 26950]

In [11]:
function σ_eq(ε_nl)
    εArray, εArray_nl = get_array.((ε_nl, ε_nl))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    Λpos = diagm(0 => max.(0, Λ))
    Λpos_nl = diagm(0 => max.(0, Λ_nl))
    εPos = TensorValue(P * Λpos * P')
    εPos_nl = TensorValue(P_nl * Λpos_nl * P_nl')
   e1 = 0.5*(Λ_nl[1]-abs(Λ_nl[1]))
    e2 = 0.5*(Λ_nl[2]-abs(Λ_nl[2]))
    e1 = min(e1,e2)
    e2 = max(e1,e2)
    tre1e2_min = 0.5*((Λ_nl[1]+Λ_nl[2])-abs(Λ_nl[1]+Λ_nl[2]))
    En_neg_temp = μ*(e2-e1)/cϕ + (λ*(tre1e2_min)+μ*(e1+e2))*tϕ
    En_neg = 0.5*(En_neg_temp + abs(En_neg_temp))
    ψPos = (0.5 * ((tr(ε_nl) >= 0) *k1* (λ * tr(ε_nl) * tr(ε_nl)) + k2*2μ * (εPos_nl ⊙ εPos_nl))) + (0.5/G)*k3*En_neg*En_neg
    return √(2 * ψPos * E)
end

σ_eq (generic function with 1 method)

In [12]:
function σ_eq1(ε_nl)
    εArray, εArray_nl = get_array.((ε_nl, ε_nl))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    Λpos = diagm(0 => max.(0, Λ))
    Λpos_nl = diagm(0 => max.(0, Λ_nl))
    εPos = TensorValue(P * Λpos * P')
    εPos_nl = TensorValue(P_nl * Λpos_nl * P_nl')
    ψPos = (0.5 * ((tr(ε_nl) >= 0) *k1* (λ * tr(ε_nl) * tr(ε_nl)) + k2*2μ * (εPos_nl ⊙ εPos_nl))) 
    return √(2 * ψPos * E)
end

function σ_eq2( ε_nl)
    εArray, εArray_nl = get_array.((ε_nl, ε_nl))
    Λ, P = eigen(εArray)
    Λ_nl, P_nl = eigen(εArray_nl)
    Λpos = diagm(0 => max.(0, Λ))
    Λpos_nl = diagm(0 => max.(0, Λ_nl))
   e1 = 0.5*(Λ_nl[1]-abs(Λ_nl[1]))
    e2 = 0.5*(Λ_nl[2]-abs(Λ_nl[2]))
    e1 = min(e1,e2)
    e2 = max(e1,e2)
    tre1e2_min = 0.5*((Λ_nl[1]+Λ_nl[2])-abs(Λ_nl[1]+Λ_nl[2])) 
    En_neg_temp = μ*(e2-e1)/cϕ + (λ*(tre1e2_min)+μ*(e1+e2))*tϕ
    En_neg = 0.5*(En_neg_temp + abs(En_neg_temp))
    ψPos = (0.5/G)*k3*En_neg*En_neg
    return √(2 * ψPos * E)
end

σ_eq2 (generic function with 1 method)

In [13]:
function G_kill(σ_eq1,σ_eq2)
    driv_eq = sqrt.((σ_eq2./coh) ⊙ (σ_eq2./coh) .+  (σ_eq1./fₜ) ⊙ (σ_eq1./fₜ) ) .- 1.0
 G_kill =  0.5 .* G0.* ( driv_eq .+ abs.(driv_eq)  )
    return G_kill
end

G_kill (generic function with 1 method)

In [14]:
function new_EnergyState(ψPlusPrev_in,ψhPos_in)
    ψPlus_in = ψhPos_in
    if ψPlus_in <= ψPlusPrev_in
        ψPlus_out = ψPlusPrev_in 
        damaged = false
    else
        ψPlus_out = ψPlus_in  
        damaged = true
    end
    damaged, ψPlus_out   
end

new_EnergyState (generic function with 1 method)

In [15]:
function barw_closed_form(D, delT1,m_p,G_k_in,σ_p)
    s = Cal_Var(σ_p,G_k_in,τ)
    μ = Cal_mean(m_p,G_k_in,τ,delT1,D,σ_p)
    λc = 1
    bar_w_nds = 1.0 ./ sqrt.(1.0 .+λc .* s) .* (exp.( - (λc .* μ .*μ ) ./ (2.0 * (1.0 .+ λc .* s))) )
    return FEFunction(Vphi, bar_w_nds .+ 1e-6), μ, s
end

function Cal_mean(m_p,G_k_in,τ,delT1,D,σ_p)
    g = -4 .* G_k_in .* delT1 .* D .* m_p
    H = - 4 .* G_k_in .* delT1 .* D 
     m_new = (m_p .- τ .* D .*g) ./ (1 .+  (τ .* D .* H )) 
    return m_new
end

Cal_mean (generic function with 1 method)

In [16]:
function Cal_Var(σ_p, G_k_in, τ)
    β = sqrt.(σ_p) ./ (τ * D)
    H = -4 .* G_k_in .* delT1 .* D
    c = 1 ./(2 .* τ .* D) .+ H ./ 2 # .+1/R
    disc = β.^2 .+ 8 .* c
    σ_new = similar(σ_p)
    pos_mask = disc .> 0
    β_pos = β[pos_mask]
    c_pos = c[pos_mask]
    y1 = (-β_pos .+ sqrt.(β_pos.^2 .+ 8 .* c_pos)) ./ 2
    y2 = (-β_pos .- sqrt.(β_pos.^2 .+ 8 .* c_pos)) ./ 2
    y_subset = max.(y1, y2)
    σ_new[pos_mask] = 1 ./ (y_subset.^2)
    neg_mask = .!pos_mask
    σ_new[neg_mask] = 1 ./ ((-β[neg_mask] ./ 2).^2)
    σ_new .= max.(σ_new, 1e-12)
    return σ_new
end


Cal_Var (generic function with 1 method)

In [17]:
using NearestNeighbors
data = zeros(2,Nₙ)
data[1,:] =nodeCoordX
data[2,:] =nodeCoordY
points = data
balltree = BallTree(data)
idxs = inrange(balltree, points, r, true)

29546-element Vector{Vector{Int64}}:
 [1]
 [2]
 [3]
 [4]
 [5, 8, 53, 54, 55, 56, 57, 58, 59, 60  …  29186, 29229, 29230, 29231, 29242, 29347, 29458, 29481, 29516, 29529]
 [6, 7, 429, 430, 431, 432, 433, 434, 435, 436  …  29156, 29165, 29173, 29178, 29309, 29351, 29421, 29431, 29486, 29503]
 [6, 7, 430, 431, 432, 433, 434, 435, 436, 437  …  29156, 29165, 29173, 29178, 29309, 29351, 29421, 29431, 29486, 29503]
 [5, 8, 53, 54, 55, 56, 57, 58, 59, 60  …  29186, 29229, 29230, 29231, 29242, 29347, 29458, 29481, 29516, 29529]
 [9]
 [10]
 [11]
 [12]
 [13]
 ⋮
 [3965, 4630, 4631, 4632, 5127, 5480, 5492, 5500, 25626, 25627, 26421, 27370, 28098, 28167, 28733, 29535]
 [1783, 2298, 2299, 2300, 2876, 3008, 3010, 3058, 3060, 3068, 3092, 26482, 28173, 28311, 28323, 28435, 28612, 28613, 28958, 29536]
 [194, 195, 196, 197, 198, 199, 200, 201, 202, 203  …  29330, 29373, 29390, 29407, 29433, 29466, 29478, 29507, 29528, 29537]
 [290, 291, 292, 293, 294, 295, 296, 297, 298, 299  …  29294, 29310, 29311, 29316

In [18]:
function nonLocalGk(G_k_prev)
    GkVec = evaluate(G_k_prev,x_S)
    caches = [array_cache(GkVec) for k in 1:Threads.nthreads()]
    Gk_nds = zeros(Nₙ)
    Gk_sum = zeros(Nₙ)
    Gk_count = zeros(Int, Nₙ)
    Threads.@threads for iel in 1:Nₑ
        cache = caches[Threads.threadid()] 
        ElNdInd = elem[iel]
    val_G = getindex!(cache, GkVec, iel)
for node in ElNdInd
        Gk_sum[node] += val_G[1]
        Gk_count[node] += 1
        end
    end
Gk_nds = Gk_sum ./ Gk_count
    return Gk_nds
end

nonLocalGk (generic function with 1 method)

In [19]:
 labels = get_face_labeling(model)
reffe_Disp = ReferenceFE(lagrangian,VectorValue{2,Float64},2)
V0_Disp = TestFESpace(model,reffe_Disp;
          conformity=:H1,
          dirichlet_tags=["BottomEdge","TopEdge"],
          dirichlet_masks=[(true,true),(false,true)]) #

UnconstrainedFESpace()

In [20]:
LoadTagId = get_tag_from_name(labels,"LeftEdge")
Γ_Load = BoundaryTriangulation(model,tags = LoadTagId)
dΓ_Load = Measure(Γ_Load,degree)
n_Γ_Load = get_normal_vector(Γ_Load)

LoadTagId2 = get_tag_from_name(labels,"RightEdge")
Γ_Load2 = BoundaryTriangulation(model,tags = LoadTagId2)
dΓ_Load2 = Measure(Γ_Load2,degree)
n_Γ_Load2 = get_normal_vector(Γ_Load2)

LoadTagId3 = get_tag_from_name(labels,"BottomEdge")
Γ_Load3 = BoundaryTriangulation(model,tags = LoadTagId3)
dΓ_Load3 = Measure(Γ_Load3,degree)
n_Γ_Load3 = get_normal_vector(Γ_Load3)

GenericCellField():
 num_cells: 13
 DomainStyle: ReferenceDomain()
 Triangulation: BoundaryTriangulation()
 Triangulation id: 10388642895734421913

In [21]:
reffephi  = ReferenceFE(lagrangian,Float64,1)
Vphi  = TestFESpace(model,reffephi;
          conformity=:H1)

UnconstrainedFESpace()

In [22]:
function step_disp(uApp,uh,m_p,σ_p,ϕ,G_k_cell)
uApp1(x) = VectorValue(0.0,0.0)
uApp2(x) = VectorValue(0.0,-uApp) 
U_Disp = TrialFESpace(V0_Disp,[uApp1,uApp2])
σ_eq_1 = σ_eq1∘((ε(uh))*(ϕ))
σ_eq_2 = σ_eq2∘((ε(uh))*(ϕ))
G_k_in = G_kill∘(σ_eq_1,σ_eq_2)
update_state!(new_EnergyState,G_k_cell,G_k_in)
G_k_nds = nonLocalGk(G_k_cell)
ϕ,m_p,σ_p = barw_closed_form(D,delT,m_p,G_k_nds,σ_p)
a(u,v) = ∫( ε(v) ⊙ ((σ∘((ε(u))))  .*(ϕ+1e-8)))dΩ
l(v) = 0.0
op = AffineFEOperator(a,l,U_Disp,V0_Disp)
ls = LUSolver()
solver = LinearFESolver(ls)
uh = solve(solver,op)
    return uh, ϕ, m_p, σ_p, G_k_cell, FEFunction(Vphi, G_k_nds),σ_eq_1, σ_eq_2 
end

step_disp (generic function with 1 method)

In [23]:
dΩ_ro = Measure(Ω,1)
x_S = get_cell_points(dΩ_ro)

CellPoint()

In [24]:
cd("UC_hessian_results") 
cd("Trial")

In [25]:
uh = zero(V0_Disp)
Tmax = 0.1
delT = 1.0e-4
vApp = 1.0
count_n = 0
m_p = zeros(Nₙ) .+ 1e-6
σ_p = zeros(Nₙ) 
T = 0.0
Load = Float64[]
Displacement = Float64[]
push!(Load, 0.0)
push!(Displacement, 0.0)
bh_cell = CellState(0.0,dΩ_ro)
bh_nd_nl = nonLocalGk(bh_cell)
uh_prev = zero(V0_Disp)
uh_in_FE = uh
f_new = 1.0
ϕ_prev = interpolate_everywhere(f_new,Vphi)
ϕ = interpolate_everywhere(f_new,Vphi)
Gk_FE = zero(Vphi)
innerMax = 10
σ_eq_1 = σ_eq1∘(ε(uh))
σ_eq_2 = σ_eq2∘(ε(uh))
start_time = time()
G_k_cell = CellState(0.0,dΩ_ro)
while T <= Tmax
    count_n = count_n + 1
T = T + delT
uApp  = T*vApp
    print("\n Entering displacemtent step :", float(uApp))
 for inner = 1:innerMax
uh,ϕ,m_p,σ_p,G_k_cell,Gk_FE, σ_eq_1, σ_eq_2 =  step_disp(uApp,uh,m_p,σ_p,ϕ,G_k_cell)
e = uh - uh_in_FE
err = sqrt(sum( ∫( e⊙e )*dΩ ))
ϕ_prev = ϕ
uh_in_FE = uh
print("\n error = ",float(err))
        if err < 1e-8
            break 
        end  
    end
Node_Force = sum(∫( n_Γ_Load ⋅ (σ∘( (ε(uh))) ).*ϕ)dΓ_Load)
push!(Load, abs(Node_Force[2]))
push!(Displacement, uApp)
 if mod(count_n,5) == 0
writevtk(Ω,"UC_Gaussian_final_$count_n",cellfields=["disp"=>uh, "phi"=>ϕ])   
    end
    end
end_time = time()
elapsed_time = end_time - start_time


 Entering displacemtent step :0.0001
 error = 0.005943805898002514
 error = 1.2137548410103793e-15
 Entering displacemtent step :0.0002
 error = 0.005943805898003386
 error = 4.915891665737325e-15
 Entering displacemtent step :0.00030000000000000003
 error = 0.005943805898004152
 error = 3.50584035600743e-15
 Entering displacemtent step :0.0004
 error = 0.005943805898003205
 error = 1.1280304565204734e-15
 Entering displacemtent step :0.0005
 error = 0.005943805898001773
 error = 3.509034521636323e-15
 Entering displacemtent step :0.0006000000000000001
 error = 0.005943805897999915
 error = 1.0900787546258663e-14
 Entering displacemtent step :0.0007000000000000001
 error = 0.005943805898009895
 error = 9.001712916369028e-15
 Entering displacemtent step :0.0008000000000000001
 error = 0.0059438058980025195
 error = 6.432938604765711e-15
 Entering displacemtent step :0.0009000000000000002
 error = 0.005943805898001321
 error = 6.668710980634085e-15
 Entering displacemtent step :0.001000

 error = 1.2193473200624506e-7
 error = 1.3359821628320014e-7
 error = 1.4569629571271514e-7
 error = 1.5811660697048984e-7
 error = 1.7073116953428408e-7
 error = 1.8340017707856214e-7
 error = 1.9597661519884987e-7
 error = 2.083115299280612e-7
 error = 2.2025897602807547e-7
 Entering displacemtent step :0.008900000000000002
 error = 0.005943829081517912
 error = 2.4837184750690837e-7
 error = 2.587599891569169e-7
 error = 2.682564854217934e-7
 error = 2.767862507816412e-7
 error = 2.843062134484677e-7
 error = 2.907831730376936e-7
 error = 2.962126623092188e-7
 error = 3.006147545582054e-7
 error = 3.040303683781177e-7
 Entering displacemtent step :0.009000000000000001
 error = 0.005943837682971587
 error = 3.158195910983505e-7
 error = 3.168779808320484e-7
 error = 3.1727788281257505e-7
 error = 3.1713917848137955e-7
 error = 3.1658654985352993e-7
 error = 3.1574637559048576e-7
 error = 3.1474400165340824e-7
 error = 3.1370451126321847e-7
 error = 3.1276327441385534e-7
 Entering di

 error = 2.0730065164845937e-6
 error = 2.033432984724616e-6
 error = 1.9944264938740683e-6
 error = 1.956083709304746e-6
 error = 1.918565655721667e-6
 error = 1.881973325404651e-6
 error = 1.8464609881808724e-6
 error = 1.812116848269757e-6
 Entering displacemtent step :0.011799999999999984
 error = 0.005944409450103032
 error = 1.776806654387186e-6
 error = 1.7466830301921763e-6
 error = 1.718659428336235e-6
 error = 1.6928377648521953e-6
 error = 1.6694619037807575e-6
 error = 1.6480598584479193e-6
 error = 1.6283487292809341e-6
 error = 1.6094356086899204e-6
 error = 1.5918206299450303e-6
 Entering displacemtent step :0.011899999999999984
 error = 0.005944411083266122
 error = 1.5936363392560228e-6
 error = 1.584729679278247e-6
 error = 1.5783976239828218e-6
 error = 1.574464311983273e-6
 error = 1.5728806895704934e-6
 error = 1.573645555586344e-6
 error = 1.5767822951853356e-6
 error = 1.5822988375923265e-6
 error = 1.5901931801070329e-6
 Entering displacemtent step :0.0119999999

 error = 6.434990376070921e-6
 error = 6.401745796855444e-6
 error = 6.370701889973572e-6
 error = 6.3393702247481565e-6
 error = 6.3101778921991264e-6
 error = 6.282879237883029e-6
 error = 6.25686101189129e-6
 error = 6.228381779073574e-6
 Entering displacemtent step :0.014699999999999967
 error = 0.005946201616502121
 error = 6.228708173234982e-6
 error = 6.193681407431348e-6
 error = 6.150401417523739e-6
 error = 6.113075528427417e-6
 error = 6.069654313984381e-6
 error = 6.0356628967265965e-6
 error = 5.994045172699029e-6
 error = 5.9554218599925375e-6
 error = 5.921603519990791e-6
 Entering displacemtent step :0.014799999999999966
 error = 0.005946231375879642
 error = 5.927800030548733e-6
 error = 5.901317737395483e-6
 error = 5.873095880802534e-6
 error = 5.84840615895305e-6
 error = 5.823107166949159e-6
 error = 5.79823228359865e-6
 error = 5.773802437172108e-6
 error = 5.750053197725031e-6
 error = 5.727418219317106e-6
 Entering displacemtent step :0.014899999999999965
 error

 error = 7.939053354914042e-6
 error = 7.98117255740416e-6
 error = 8.026834412808182e-6
 error = 8.076096578310384e-6
 Entering displacemtent step :0.01759999999999995
 error = 0.00594819574374789
 error = 8.26331951969708e-6
 error = 8.323789988962658e-6
 error = 8.388510562158527e-6
 error = 8.456859448340495e-6
 error = 8.528670590305721e-6
 error = 8.606068093728673e-6
 error = 8.683071249724209e-6
 error = 8.762325763432184e-6
 error = 8.85047216270304e-6
 Entering displacemtent step :0.01769999999999995
 error = 0.005948348657983809
 error = 9.088654895720065e-6
 error = 9.187374056862535e-6
 error = 9.277795935594757e-6
 error = 9.373267838459888e-6
 error = 9.483759816054762e-6
 error = 9.579197604696833e-6
 error = 9.672159536401842e-6
 error = 9.762043681199761e-6
 error = 9.869995033119449e-6
 Entering displacemtent step :0.017799999999999948
 error = 0.005948535677254024
 error = 1.0144733087908837e-5
 error = 1.0234050541166786e-5
 error = 1.0342222393230866e-5
 error = 1

...

 error = 0.0025013008598695494
 error = 0.0024981398294144728
 error = 0.002497195286431849
 error = 0.0024940381425121406
 error = 0.002493554896019762
 Entering displacemtent step :0.08990000000000155
 error = 0.0070684155508099105
 error = 0.002493927076407824
 error = 0.0024892509619691976
 error = 0.0024897120244659772
 error = 0.0024860311132012623
 error = 0.0024850679277382444
 error = 0.002486019500339954
 error = 0.0024831130227137123
 error = 0.002482648874850086
 error = 0.002479951982369894
 Entering displacemtent step :0.09000000000000155
 error = 0.007000540301282056
 error = 0.00247971191218871
 error = 0.0024820094352398458
 error = 0.0024787265923525443
 error = 0.002475519893719858
 error = 0.0024854478005970743
 error = 0.002478905352588746
 error = 0.002448809352544662
 error = 0.002482057555749509
 error = 0.002475904849024939
 Entering displacemtent step :0.09010000000000155
 error = 0.006951380948087365
 error = 0.002478834103083389
 error = 0.002480952679688907

 error = 0.0028163037239141513
 error = 0.0028504925037825786
 error = 0.0027785199996812975
 error = 0.002815200617445421
 error = 0.002805096435029675
 error = 0.002843389974452499
 Entering displacemtent step :0.09280000000000163
 error = 0.006590397290767423
 error = 0.0028931023677506804
 error = 0.0028179899090246254
 error = 0.002840095442662132
 error = 0.0028001242281787374
 error = 0.0028664680989555985
 error = 0.0030277492289900914
 error = 0.002913720063816684
 error = 0.002921579462317905
 error = 0.002942681382780914
 Entering displacemtent step :0.09290000000000163
 error = 0.006620160900006849
 error = 0.002958028698423478
 error = 0.0029816855650135676
 error = 0.002986309967893061
 error = 0.0030104789229168766
 error = 0.00298160445133582
 error = 0.0029857885081313726
 error = 0.003010025053845143
 error = 0.003016938926087705
 error = 0.003029591908190265
 Entering displacemtent step :0.09300000000000164
 error = 0.006632025325956095
 error = 0.003061447300003089


 error = 0.006230614786503993
 error = 0.0062669205460847395
 error = 0.007245293269431758
 error = 0.00634520219479159
 error = 0.006383540004442084
 Entering displacemtent step :0.09570000000000171
 error = 0.008783680399485006
 error = 0.006467976893621038
 error = 0.006506430223632596
 error = 0.006544766637875607
 error = 0.006582470064477904
 error = 0.006619006777069114
 error = 0.006654721066984259
 error = 0.006688435828488363
 error = 0.006721167015100512
 error = 0.006751613992357
 Entering displacemtent step :0.09580000000000172
 error = 0.008388821829224728
 error = 0.00580516864379717
 error = 0.005845236475308629
 error = 0.005882283886217632
 error = 0.00591849848409466
 error = 0.005948968003350539
 error = 0.005975556328180352
 error = 0.00599836406112033
 error = 0.006016863019438525
 error = 0.0060314258300352514
 Entering displacemtent step :0.09590000000000172
 error = 0.00860876769110903
 error = 0.006053168207506721
 error = 0.006053645644098019
 error = 0.00604

 error = 0.0007332088171543925
 error = 0.0007337224502461722
 error = 0.0007356299224311987
 error = 0.0007348258232766555
 error = 0.0007412670692936662
 Entering displacemtent step :0.0986000000000018
 error = 0.006534984585671421
 error = 0.0007417043854258665
 error = 0.0007386442163193474
 error = 0.0007425833871709947
 error = 0.0007395176659459393
 error = 0.000743704173858492
 error = 0.0007404277989587501
 error = 0.0007445759750476703
 error = 0.000741401809848977
 error = 0.0007465928988142558
 Entering displacemtent step :0.0987000000000018
 error = 0.006536270037887119
 error = 0.0007474623868594704
 error = 0.0007487402112516133
 error = 0.0007491960697814608
 error = 0.0007496493075770598
 error = 0.000750107814197847
 error = 0.0007505825013287865
 error = 0.0007510434815206517
 error = 0.0007510078956327401
 error = 0.0007520209900123513
 Entering displacemtent step :0.0988000000000018
 error = 0.006537701277926333
 error = 0.000753728452569487
 error = 0.000754235458

40359.94799995422